# Motion potential 2D — training

This notebook generates the dataset and trains `TrajectoryPredictor`. The best checkpoint is written to `best_model.pt`. Investigation / plotting lives in a separate notebook (`investigate.ipynb`) so we don't have to re-train when we want to poke at outputs.

All logic lives in three modules:
- `data_generation.py` — configs + potential sampling + Verlet integrator + `generate_dataset`
- `models.py` — `ParticleTrajectoryDataset`, `PotentialEncoder`, `TrajectoryPredictor`, `masked_mse_loss`
- `train.py` — `TrainConfig`, `run_training`, loop helpers

## Colab setup

If running in Colab, mount Drive and add the folder with the `.py` files to `sys.path`. Skip this cell when running locally next to the `.py` files.

In [ ]:
# Colab-only. Adjust PROJECT_DIR to where you put the .py files in Drive.
# from google.colab import drive
# drive.mount('/content/drive')
# import sys
# PROJECT_DIR = '/content/drive/MyDrive/motion_potential_2d'
# if PROJECT_DIR not in sys.path:
#     sys.path.insert(0, PROJECT_DIR)
# %cd $PROJECT_DIR

In [ ]:
# Auto-reload edited .py files without restarting the kernel.
%load_ext autoreload
%autoreload 2

## 1. Generate the dataset

Produces `particle_2d_dataset.npz`. Skip this cell if the file already exists and you just want to re-train.

In [ ]:
from data_generation import GridConfig, TrajConfig, DataConfig, generate_dataset

grid_cfg = GridConfig(xlim=4.0, ylim=4.0, nx=64, ny=64)
traj_cfg = TrajConfig(dt=0.01, steps=2000, mass=0.2)
data_cfg = DataConfig(
    offset_range=(-5.0, 5.0),
    central_prob=0.25,
    arbitrary_prob=0.75,
    fourier_prob=0.5,
    gaussian_bump_prob=0.5,
    max_fourier_mode=4,
    fourier_alpha=3.0,
    min_bumps=4,
    max_bumps=9,
    max_init_speed=2.5,
)

generate_dataset(
    path="particle_2d_dataset.npz",
    n_examples=2000,
    seed=42,
    grid_cfg=grid_cfg,
    traj_cfg=traj_cfg,
    data_cfg=data_cfg,
)

## 2. Quick peek at a generated example

Sanity check that the potential / trajectory look reasonable before training.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

data = np.load("particle_2d_dataset.npz")

idx = 2
V = data["potentials"][idx]
mask = data["traj_mask"][idx].astype(bool)
traj = data["trajectories"][idx][mask]
Lz = data["angular_momenta"][idx][mask]
xs = data["xs"]
ys = data["ys"]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

im = axes[0].imshow(V.T, origin="lower", extent=[xs[0], xs[-1], ys[0], ys[-1]], aspect="auto")
plt.colorbar(im, ax=axes[0], label="V(x,y)")
axes[0].set(xlabel="x", ylabel="y", title=f"Potential #{idx}")

axes[1].plot(traj[:, 0], traj[:, 1])
axes[1].set(xlabel="x", ylabel="y", title=f"Trajectory #{idx}")
axes[1].axis("equal")

axes[2].plot(Lz)
axes[2].set(xlabel="time step", ylabel="Lz", title=f"Angular momentum $L_z(t)$ #{idx}")

plt.tight_layout()
plt.show()

## 3. Train

`run_training` builds the dataset splits, creates the model, trains it, and saves the best-val checkpoint to `best_model.pt`.

In [ ]:
from train import TrainConfig, run_training

train_cfg = TrainConfig(
    batch_size=32,
    num_epochs=30,
    lr=1e-3,
    seed=42,
    train_frac=0.8,
    val_frac=0.1,
    num_workers=0,
)

model, history = run_training(
    npz_path="particle_2d_dataset.npz",
    checkpoint_path="best_model.pt",
    traj_len=2001,
    state_dim=4,
    pot_latent_dim=256,
    hidden_dim=512,
    train_cfg=train_cfg,
)

## 4. Loss curves

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(history["train_loss"]) + 1)
plt.figure(figsize=(7, 4))
plt.plot(epochs, history["train_loss"], label="train")
plt.plot(epochs, history["val_loss"], label="val")
plt.xlabel("epoch")
plt.ylabel("masked MSE")
plt.yscale("log")
plt.title("Training curves")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.show()

print(f"Best val loss: {history['best_val_loss']:.6f}")

## 5. Test-set loss

Evaluate on the held-out test split using the same split seed as training.

In [ ]:
import torch
from torch.utils.data import DataLoader

from models import ParticleTrajectoryDataset
from train import build_splits, eval_one_epoch

device = next(model.parameters()).device

dataset = ParticleTrajectoryDataset("particle_2d_dataset.npz")
_, _, test_set = build_splits(
    dataset,
    train_frac=train_cfg.train_frac,
    val_frac=train_cfg.val_frac,
    seed=train_cfg.seed,
)
test_loader = DataLoader(test_set, batch_size=train_cfg.batch_size, shuffle=False)

test_loss = eval_one_epoch(model, test_loader, device)
print("Test loss:", test_loss)

At this point `best_model.pt` and `particle_2d_dataset.npz` are on disk. Open `investigate.ipynb` next to dig into predictions without re-training.